[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/093.%20The%20Inventory-Routing%20Problem%20%28IRP%29/P93-Tier-6_executed.ipynb)

---

# 93. The Inventory-Routing Problem (IRP)
## Tier 6: Multi-Agent Coordination

### Purpose and scope

This notebook builds a small agent-based simulation for an inventory-routing setting. Customer agents monitor local stock and create replenishment tasks. Vehicle agents submit cost-based bids for those tasks. A market coordinates task allocation through a lowest-price auction subject to a confidence threshold, and a depot agent checks whether requested stock is available.

The example instance contains one depot, 20 customers, six vehicles, and a 10-period planning horizon. The code reports simulated inventory, deliveries, costs, service, and market activity. It illustrates the specified multi-agent workflow; it does **not** solve a centralized optimization model or establish that decentralized coordination is globally optimal.

### How the coordination mechanism works

Each period follows the same sequence:

1. Customer agents experience demand and request replenishment when stock falls below a threshold.
2. Every vehicle with sufficient remaining capacity calculates a bid for each active task. A bid estimates travel and service costs, completion time, and confidence.
3. The market awards each task to the lowest-priced bid that exceeds the confidence threshold. If no bid qualifies, the task is not awarded.
4. The depot checks stock availability. A successful delivery reduces depot inventory and increases the customer's inventory.
5. Vehicles return to the depot, and the simulation records period and cumulative performance.

The mechanism is decentralized in the sense that agents make local decisions and the market coordinates bids. The implementation runs sequentially in one Python process; it does not model distributed hardware, communication failures, learning agents, or an explicit Contract Net Protocol.

### Formal description of the auction rule

For a task \(j\), let \(B_j\) be the bids submitted by vehicles that can carry the task and meet its deadline. Let \(p_{ij}\) be vehicle \(i\)'s bid price and \(q_{ij}\) its confidence. The eligible bids are

$$
E_j = \{i \in B_j : q_{ij} > 0.6\}.
$$

If \(E_j\) is nonempty, the market selects

$$
i^*(j) \in \underset{i \in E_j}{\operatorname{argmin}}\; p_{ij}.
$$

Otherwise, task \(j\) receives no award. If several eligible bids have the same lowest price, the first one in the submitted bid order wins. This is a task-by-task auction, not a joint optimization over all routes. With \(T\) tasks and \(V\) vehicles, calculating bids requires \(O(TV)\) bid evaluations; this bound does not include the simulation's inventory and plotting work.

### Modeling assumptions

- Locations are two-dimensional coordinates, and travel distance is Euclidean distance in coordinate units.
- Vehicles travel at 50 distance units per hour and spend one hour servicing each delivery. Each task has a 24-hour deadline measured from the start of its bidding period.
- A vehicle starts each period at the depot. Within a period, its next bid uses its current location and remaining capacity; it returns to the depot after task allocation.
- Demand is sampled from a normal distribution with the customer-specific mean and standard deviation. Negative sampled demand is set to zero. Demand exceeding on-hand inventory is recorded as a stockout event, and inventory is floored at zero.
- Customer deliveries are capped at the customer's maximum inventory. The depot has a finite initial stock and receives no replenishment during this example.
- The vehicle bid includes travel cost and an amortized share of fixed cost. The simulation records the corresponding travel and per-task amortized cost when the task is executed.
- Holding cost is charged on end-of-period inventory. The reported system revenue is the reward attached to completed customer tasks; internal depot-to-customer transfer revenue is not counted again as system revenue.
- Market prices and agent decisions are illustrative rules, not calibrated economic policies. Results depend on the sampled demand and the specified assumptions.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Task:
    """A customer replenishment request offered to the vehicle agents."""

    task_id: str
    customer_id: int
    quantity: float
    deadline: float
    priority: int
    reward: float
    location: Tuple[float, float]


@dataclass
class Bid:
    """A vehicle's estimated cost and service confidence for one task."""

    vehicle_id: int
    task_id: str
    bid_price: float
    completion_time: float
    confidence: float


@dataclass
class Agent:
    """Shared identifying information for customer, vehicle, and depot agents."""

    agent_id: str
    agent_type: str
    location_id: int
    inventory: float
    capabilities: Dict[str, float]


class CustomerAgent(Agent):
    """Monitor customer inventory, experience demand, and request replenishment."""

    def __init__(self, agent_id: str, customer_data, market):
        super().__init__(
            agent_id=agent_id,
            agent_type="customer",
            location_id=customer_data["id"],
            inventory=customer_data["initial_inventory"],
            capabilities={"demand_rate": customer_data["demand_mean"]},
        )
        self.min_inventory = customer_data["min_inventory"]
        self.max_inventory = customer_data["max_inventory"]
        self.holding_cost = customer_data["holding_cost"]
        self.demand_std = customer_data["demand_std"]
        self.location = (customer_data["x"], customer_data["y"])
        self.market = market
        self.demand_history = []
        self.stockout_events = 0
        self._request_number = 0

    def check_inventory_status(self) -> bool:
        """Request replenishment below 1.5 times the minimum stock level."""
        return self.inventory < self.min_inventory * 1.5

    def generate_replenishment_request(self) -> Optional[Task]:
        """Create a task when stock is low and there is room to receive stock."""
        if not self.check_inventory_status():
            return None

        required_quantity = min(self.max_inventory - self.inventory, 50.0)
        if required_quantity <= 0:
            return None

        if self.inventory < self.min_inventory:
            priority = 1
        elif self.inventory < self.min_inventory * 1.2:
            priority = 2
        else:
            priority = 3

        base_reward = required_quantity * 10.0
        urgency_multiplier = 1 + (4 - priority) * 0.2
        self._request_number += 1
        return Task(
            task_id=f"REQ_{self.agent_id}_{self._request_number}",
            customer_id=self.location_id,
            quantity=required_quantity,
            deadline=24.0,
            priority=priority,
            reward=base_reward * urgency_multiplier,
            location=self.location,
        )

    def consume_demand(self) -> float:
        """Apply one period of nonnegative demand and record any stockout event."""
        demand = max(
            0.0,
            float(
                np.random.normal(
                    self.capabilities["demand_rate"], self.demand_std
                )
            ),
        )
        if demand > self.inventory:
            self.stockout_events += 1
        self.inventory = max(0.0, self.inventory - demand)
        self.demand_history.append(demand)
        return demand

    def receive_delivery(self, quantity: float) -> None:
        """Add delivered stock without exceeding the customer's storage limit."""
        self.inventory = min(self.inventory + quantity, self.max_inventory)


class VehicleAgent(Agent):
    """Estimate task bids and execute awarded deliveries within vehicle capacity."""

    def __init__(self, agent_id: str, vehicle_data, market):
        super().__init__(
            agent_id=agent_id,
            agent_type="vehicle",
            location_id=0,
            inventory=0.0,
            capabilities={"capacity": vehicle_data["capacity"]},
        )
        self.capacity = vehicle_data["capacity"]
        self.fixed_cost = vehicle_data["fixed_cost"]
        self.variable_cost = vehicle_data["variable_cost"]
        self.current_location = (0.0, 0.0)
        self.market = market
        self.current_route = []
        self.route_history = []
        self.available_capacity = self.capacity
        self.total_earnings = 0.0
        self.total_costs = 0.0
        self.total_quantity_delivered = 0.0
        self.total_tasks_completed = 0
        self._period_quantity = 0.0

    def calculate_bid(self, task: Task) -> Optional[Bid]:
        """Estimate the cost, completion time, and confidence for a task."""
        if task.quantity > self.available_capacity:
            return None

        distance = float(
            np.hypot(
                self.current_location[0] - task.location[0],
                self.current_location[1] - task.location[1],
            )
        )
        travel_time = distance / 50.0
        service_time = 1.0
        completion_time = travel_time + service_time
        if completion_time > task.deadline:
            return None

        travel_cost = distance * self.variable_cost
        service_cost = self.fixed_cost / 20.0
        total_cost = travel_cost + service_cost
        bid_price = total_cost * 1.2

        capacity_utilization = (self.capacity - self.available_capacity) / self.capacity
        urgency_factor = (6 - task.priority) / 5
        confidence = max(
            0.5,
            1 - capacity_utilization * 0.3 - urgency_factor * 0.2,
        )
        return Bid(
            vehicle_id=int(self.agent_id.split("_")[1]),
            task_id=task.task_id,
            bid_price=bid_price,
            completion_time=completion_time,
            confidence=confidence,
        )

    def execute_task(self, task: Task) -> float:
        """Travel to the customer and account for the costs used in the bid."""
        distance = float(
            np.hypot(
                self.current_location[0] - task.location[0],
                self.current_location[1] - task.location[1],
            )
        )
        travel_cost = distance * self.variable_cost
        task_cost = travel_cost + self.fixed_cost / 20.0

        self.current_location = task.location
        self.available_capacity -= task.quantity
        self.total_costs += task_cost
        self.total_earnings += task.reward
        self.total_quantity_delivered += task.quantity
        self.total_tasks_completed += 1
        self._period_quantity += task.quantity
        self.current_route.append(task.customer_id)
        return task_cost

    def return_to_depot(self) -> float:
        """Record the return leg and prepare the vehicle for the next period."""
        distance = float(np.hypot(*self.current_location))
        return_cost = distance * self.variable_cost
        self.total_costs += return_cost
        self.current_location = (0.0, 0.0)
        self.route_history.append(self.current_route.copy())
        self.current_route = []
        self.available_capacity = self.capacity
        self._period_quantity = 0.0
        return return_cost


class DepotAgent(Agent):
    """Manage finite depot inventory and set an illustrative internal price."""

    def __init__(self, agent_id: str, depot_data, market):
        super().__init__(
            agent_id=agent_id,
            agent_type="depot",
            location_id=0,
            inventory=depot_data["initial_inventory"],
            capabilities={
                "storage_capacity": depot_data["initial_inventory"] * 2
            },
        )
        self.holding_cost = depot_data["holding_cost"]
        self.market = market
        self.unit_cost = 5.0
        self.total_revenue = 0.0
        self.total_costs = 0.0

    def set_internal_price(self, demand_level: float) -> float:
        """Set a price that rises with demand and falls with relative stock."""
        inventory_ratio = (
            self.inventory / self.capabilities["storage_capacity"]
        )
        inventory_factor = 2 - inventory_ratio
        demand_factor = 1 + demand_level * 0.5
        return self.unit_cost * inventory_factor * demand_factor

    def fulfill_delivery(self, quantity: float, price: float) -> bool:
        """Transfer stock only when the depot can fulfill the entire request."""
        if quantity > self.inventory:
            return False
        self.inventory -= quantity
        self.total_revenue += quantity * price
        self.total_costs += quantity * self.unit_cost
        return True


class Market:
    """Post tasks and allocate each task through a confidence-filtered auction."""

    def __init__(self):
        self.active_tasks = []
        self.completed_tasks = []
        self.task_history = []
        self.market_price = 10.0
        self.period_prices = []

    def post_task(self, task: Task) -> None:
        self.active_tasks.append(task)
        self.task_history.append(("POST", task.task_id))

    def get_tasks(self) -> List[Task]:
        return self.active_tasks.copy()

    def auction_task(self, task: Task, bids: List[Bid]) -> Optional[Bid]:
        """Award the lowest bid among bids whose confidence exceeds 0.6."""
        eligible_bids = [bid for bid in bids if bid.confidence > 0.6]
        if not eligible_bids:
            return None
        return min(eligible_bids, key=lambda bid: bid.bid_price)

    def complete_task(self, task: Task, winning_bid: Bid) -> None:
        if task in self.active_tasks:
            self.active_tasks.remove(task)
        self.completed_tasks.append(task)
        self.task_history.append(("COMPLETE", task.task_id))
        price_per_unit = winning_bid.bid_price / task.quantity
        self.market_price = 0.9 * self.market_price + 0.1 * price_per_unit


class MultiAgentIRP:
    """Run the customer-request, vehicle-auction, and depot-fulfillment cycle."""

    def __init__(self, depot, customers, vehicles, num_periods):
        self.depot = depot
        self.customers = customers
        self.vehicles = vehicles
        self.num_periods = num_periods
        self.market = Market()
        self.customer_agents = []
        self.vehicle_agents = []
        self.depot_agent = None
        self.total_cost = 0.0
        self.total_revenue = 0.0
        self.service_level = 0.0
        self.stockout_events = 0
        self.request_count = 0
        self.holding_costs = 0.0
        self.period_records = []
        self._initialize_agents()

    def _initialize_agents(self) -> None:
        for i, customer in enumerate(self.customers):
            data = {
                "id": customer.id,
                "x": customer.x,
                "y": customer.y,
                "initial_inventory": customer.initial_inventory,
                "min_inventory": customer.min_inventory,
                "max_inventory": customer.max_inventory,
                "demand_mean": customer.demand_mean,
                "demand_std": customer.demand_std,
                "holding_cost": customer.holding_cost,
            }
            self.customer_agents.append(
                CustomerAgent(f"CUSTOMER_{i + 1}", data, self.market)
            )

        for i, vehicle in enumerate(self.vehicles):
            data = {
                "capacity": vehicle.capacity,
                "fixed_cost": vehicle.fixed_cost,
                "variable_cost": vehicle.variable_cost,
            }
            self.vehicle_agents.append(
                VehicleAgent(f"VEHICLE_{i + 1}", data, self.market)
            )

        depot_data = {
            "initial_inventory": self.depot.initial_inventory,
            "holding_cost": self.depot.holding_cost,
        }
        self.depot_agent = DepotAgent("DEPOT_1", depot_data, self.market)
        print(
            f"Initialized {len(self.customer_agents)} customer agents, "
            f"{len(self.vehicle_agents)} vehicle agents, and 1 depot agent."
        )

    def simulate_period(self, period: int) -> None:
        print(f"\n=== PERIOD {period + 1} ===")
        print("Phase 1: Customers consume demand and post replenishment requests.")
        for customer_agent in self.customer_agents:
            customer_agent.consume_demand()
            task = customer_agent.generate_replenishment_request()
            if task is not None:
                self.market.post_task(task)
                self.request_count += 1
                print(
                    f"  {customer_agent.agent_id}: "
                    f"requested {task.quantity:.1f} units."
                )

        tasks_this_period = self.market.get_tasks()
        print("\nPhase 2: Vehicles bid; the market allocates eligible tasks.")
        for task in tasks_this_period:
            bids = []
            for vehicle_agent in self.vehicle_agents:
                bid = vehicle_agent.calculate_bid(task)
                if bid is not None:
                    bids.append(bid)
                    print(
                        f"  {vehicle_agent.agent_id}: bid ${bid.bid_price:.2f}, "
                        f"confidence {bid.confidence:.2f}."
                    )

            winning_bid = self.market.auction_task(task, bids)
            if winning_bid is None:
                print(f"  No eligible bid for task {task.task_id}.")
                continue

            winning_vehicle = next(
                vehicle
                for vehicle in self.vehicle_agents
                if int(vehicle.agent_id.split("_")[1]) == winning_bid.vehicle_id
            )
            internal_price = self.depot_agent.set_internal_price(
                len(tasks_this_period) / max(1, len(self.customer_agents))
            )
            if not self.depot_agent.fulfill_delivery(
                task.quantity, internal_price
            ):
                print(
                    f"  Depot stock is insufficient for customer "
                    f"{task.customer_id}; the task is not executed."
                )
                continue

            winning_vehicle.execute_task(task)
            customer_agent = next(
                customer
                for customer in self.customer_agents
                if customer.location_id == task.customer_id
            )
            customer_agent.receive_delivery(task.quantity)
            self.market.complete_task(task, winning_bid)
            print(
                f"  {winning_vehicle.agent_id} delivered "
                f"{task.quantity:.1f} units to customer {task.customer_id}."
            )

        # Unawarded and unfulfilled requests expire at the end of their bidding period.
        self.market.active_tasks = []

        print("\nPhase 3: Vehicles return to the depot.")
        for vehicle_agent in self.vehicle_agents:
            return_cost = vehicle_agent.return_to_depot()
            print(
                f"  {vehicle_agent.agent_id}: return cost ${return_cost:.2f}."
            )

        self._calculate_period_metrics(period)

    def _calculate_period_metrics(self, period: int) -> None:
        period_holding_cost = sum(
            customer.inventory * customer.holding_cost
            for customer in self.customer_agents
        ) + self.depot_agent.inventory * self.depot_agent.holding_cost
        self.holding_costs += period_holding_cost
        vehicle_costs = sum(
            vehicle.total_costs for vehicle in self.vehicle_agents
        )
        self.total_cost = (
            vehicle_costs + self.depot_agent.total_costs + self.holding_costs
        )
        self.total_revenue = sum(
            task.reward for task in self.market.completed_tasks
        )
        completed = len(self.market.completed_tasks)
        self.service_level = (
            100.0 * completed / self.request_count
            if self.request_count
            else 0.0
        )
        self.stockout_events = sum(
            customer.stockout_events for customer in self.customer_agents
        )
        self.market.period_prices.append(self.market.market_price)
        self.period_records.append(
            {
                "period": period + 1,
                "requests": self.request_count,
                "completed_tasks": completed,
                "stockout_events": self.stockout_events,
                "total_cost": self.total_cost,
                "total_revenue": self.total_revenue,
                "service_level": self.service_level,
                "depot_inventory": self.depot_agent.inventory,
                "average_customer_inventory": float(
                    np.mean([c.inventory for c in self.customer_agents])
                ),
                "market_price": self.market.market_price,
            }
        )

    def run_simulation(self) -> Dict[str, float]:
        print(f"Starting the multi-agent IRP simulation for {self.num_periods} periods.")
        for period in range(self.num_periods):
            self.simulate_period(period)
            print(f"\nPeriod {period + 1} summary:")
            print(f"  Cumulative cost: ${self.total_cost:.2f}")
            print(f"  Cumulative task-reward revenue: ${self.total_revenue:.2f}")
            print(f"  Cumulative service level: {self.service_level:.1f}%")
            print(f"  Stockout events: {self.stockout_events}")
            print(f"  Depot inventory: {self.depot_agent.inventory:.1f} units")

        results = {
            "total_cost": self.total_cost,
            "total_revenue": self.total_revenue,
            "service_level": self.service_level,
            "stockout_events": self.stockout_events,
            "net_profit": self.total_revenue - self.total_cost,
            "completed_tasks": len(self.market.completed_tasks),
            "request_count": self.request_count,
        }
        assert results["completed_tasks"] <= results["request_count"]
        assert np.isclose(
            results["net_profit"],
            results["total_revenue"] - results["total_cost"],
        )
        print("\n=== SIMULATION COMPLETE ===")
        print(f"Final service level: {self.service_level:.1f}%")
        print(f"Total stockout events: {self.stockout_events}")
        print(f"Net profit under the stated accounting convention: ${results['net_profit']:.2f}")
        return results

In [ ]:
def create_mas_instance():
    """Build the example depot, customer, and vehicle data."""
    depot = type(
        "Depot",
        (),
        {"x": 0, "y": 0, "initial_inventory": 1500, "holding_cost": 0.1},
    )()

    # The seed makes the generated customer locations and parameters reproducible.
    np.random.seed(42)
    customers = []
    for i in range(20):
        cluster_center_x = np.random.choice([-25, 0, 25])
        cluster_center_y = np.random.choice([-25, 0, 25])
        x = cluster_center_x + np.random.normal(0, 10)
        y = cluster_center_y + np.random.normal(0, 10)
        customer = type(
            "Customer",
            (),
            {
                "id": i + 1,
                "x": x,
                "y": y,
                "demand_mean": np.random.uniform(8, 20),
                "demand_std": np.random.uniform(2, 4),
                "holding_cost": np.random.uniform(0.1, 0.3),
                "initial_inventory": np.random.uniform(30, 70),
                "min_inventory": np.random.uniform(8, 16),
                "max_inventory": np.random.uniform(80, 120),
            },
        )()
        customers.append(customer)

    vehicles = [
        type(
            "Vehicle",
            (),
            {"id": i, "capacity": 80, "fixed_cost": 100, "variable_cost": 1.5},
        )()
        for i in range(1, 7)
    ]
    return depot, customers, vehicles


print("Creating the multi-agent IRP instance.")
depot, customers, vehicles = create_mas_instance()
print(f"Depot location: ({depot.x}, {depot.y}); initial stock: {depot.initial_inventory} units.")
print("First five customers:")
for customer in customers[:5]:
    print(
        f"  C{customer.id}: ({customer.x:.1f}, {customer.y:.1f}), "
        f"mean demand {customer.demand_mean:.1f} units, "
        f"initial stock {customer.initial_inventory:.1f} units."
    )
print(f"Vehicles: {len(vehicles)}; capacity per vehicle: {vehicles[0].capacity} units.")

In [ ]:
def run_mas_simulation():
    """Initialize the agents and run the stated 10-period example."""
    mas = MultiAgentIRP(depot, customers, vehicles, num_periods=10)
    print("\n=== MULTI-AGENT SYSTEM SIMULATION ===")
    print(f"Customer agents: {len(mas.customer_agents)}")
    print(f"Vehicle agents: {len(mas.vehicle_agents)}")
    print("Depot agents: 1")
    results = mas.run_simulation()
    return mas, results


mas, results = run_mas_simulation()

In [ ]:
def analyze_mas_performance(mas, results):
    """Report observed agent and system metrics from this simulation run."""
    print("\n=== MULTI-AGENT SYSTEM ANALYSIS ===")
    print("Vehicle agents:")
    for vehicle in mas.vehicle_agents:
        utilization = (
            100.0 * vehicle.total_quantity_delivered
            / (vehicle.capacity * mas.num_periods)
        )
        print(
            f"  {vehicle.agent_id}: earned ${vehicle.total_earnings:.2f}, "
            f"incurred ${vehicle.total_costs:.2f}, "
            f"completed {vehicle.total_tasks_completed} tasks, "
            f"period-capacity utilization {utilization:.1f}%."
        )

    inventories = [customer.inventory for customer in mas.customer_agents]
    print("\nSystem metrics:")
    print(f"  Requests: {results['request_count']}")
    print(f"  Completed tasks: {results['completed_tasks']}")
    print(f"  Service level: {results['service_level']:.1f}%")
    print(f"  Customer-demand stockout events: {results['stockout_events']}")
    print(f"  Total cost: ${results['total_cost']:.2f}")
    print(f"  Task-reward revenue: ${results['total_revenue']:.2f}")
    print(f"  Final depot inventory: {mas.depot_agent.inventory:.1f} units")
    print(f"  Mean final customer inventory: {np.mean(inventories):.1f} units")
    print(f"  Final market price indicator: ${mas.market.market_price:.2f} per unit")
    print(
        "  The market price indicator is an exponentially smoothed bid-price "
        "measure; it is not the customer task reward."
    )

    return {
        "average_vehicle_profit": float(
            np.mean([
                vehicle.total_earnings - vehicle.total_costs
                for vehicle in mas.vehicle_agents
            ])
        ),
        "average_period_capacity_utilization": float(
            np.mean(
                [
                    vehicle.total_quantity_delivered
                    / (vehicle.capacity * mas.num_periods)
                    for vehicle in mas.vehicle_agents
                ]
            )
        ),
        "average_final_customer_inventory": float(np.mean(inventories)),
        "market_price": float(mas.market.market_price),
        "service_level": float(results["service_level"]),
    }


performance_metrics = analyze_mas_performance(mas, results)

In [ ]:
def visualize_mas_results(mas, results, performance_metrics):
    """Plot geographic, inventory, market, and performance data from the run."""
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    fig.suptitle("Multi-Agent IRP Simulation Results", fontsize=16, fontweight="bold")

    ax = axes[0, 0]
    ax.scatter(0, 0, c="red", s=180, marker="s", label="Depot")
    for customer in customers:
        ax.scatter(customer.x, customer.y, c="blue", s=35, label="Customer")
        ax.annotate(f"C{customer.id}", (customer.x, customer.y), xytext=(3, 3), textcoords="offset points", fontsize=7)
    ax.set_title("Depot and Customer Locations")
    ax.set_xlabel("X coordinate (distance units)")
    ax.set_ylabel("Y coordinate (distance units)")
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys())
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal")

    vehicle_labels = [vehicle.agent_id.replace("VEHICLE_", "V") for vehicle in mas.vehicle_agents]
    vehicle_profits = [
        vehicle.total_earnings - vehicle.total_costs
        for vehicle in mas.vehicle_agents
    ]
    utilizations = [
        100 * vehicle.total_quantity_delivered
        / (vehicle.capacity * mas.num_periods)
        for vehicle in mas.vehicle_agents
    ]
    x = np.arange(len(vehicle_labels))
    width = 0.38
    ax = axes[0, 1]
    ax.bar(x - width / 2, vehicle_profits, width, label="Vehicle earnings minus costs ($)")
    ax.set_title("Vehicle Financial Result")
    ax.set_xlabel("Vehicle")
    ax.set_ylabel("Amount ($)")
    ax.set_xticks(x, vehicle_labels)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()

    ax = axes[0, 2]
    customer_labels = [f"C{customer.location_id}" for customer in mas.customer_agents]
    ax.bar(customer_labels, [c.inventory for c in mas.customer_agents], label="Final inventory")
    ax.plot(customer_labels, [c.min_inventory for c in mas.customer_agents], color="red", marker="o", label="Minimum inventory")
    ax.set_title("Final Customer Inventory")
    ax.set_xlabel("Customer")
    ax.set_ylabel("Inventory (units)")
    ax.tick_params(axis="x", rotation=60)
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)

    periods = [record["period"] for record in mas.period_records]
    ax = axes[1, 0]
    ax.plot(periods, mas.market.period_prices, marker="o", color="purple")
    ax.set_title("Observed Market Price Indicator")
    ax.set_xlabel("Period")
    ax.set_ylabel("Smoothed bid price per unit ($)")
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    ax.plot(periods, [r["service_level"] for r in mas.period_records], marker="o", label="Cumulative service level")
    ax.set_title("Cumulative Request Service")
    ax.set_xlabel("Period")
    ax.set_ylabel("Completed requests (%)")
    ax.set_ylim(0, 105)
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1, 2]
    ax.plot(periods, [r["depot_inventory"] for r in mas.period_records], marker="o", label="Depot")
    ax.plot(periods, [r["average_customer_inventory"] for r in mas.period_records], marker="s", label="Mean customer inventory")
    ax.set_title("Inventory Over Time")
    ax.set_xlabel("Period")
    ax.set_ylabel("Inventory (units)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[2, 0]
    completed_by_period = []
    previous_completed = 0
    for record in mas.period_records:
        completed_by_period.append(record["completed_tasks"] - previous_completed)
        previous_completed = record["completed_tasks"]
    ax.bar(periods, completed_by_period, color="green", label="Completed tasks")
    ax.set_title("Tasks Completed by Period")
    ax.set_xlabel("Period")
    ax.set_ylabel("Number of tasks")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)

    ax = axes[2, 1]
    ax.plot(periods, [r["total_cost"] for r in mas.period_records], marker="o", label="Cumulative cost")
    ax.plot(periods, [r["total_revenue"] for r in mas.period_records], marker="s", label="Cumulative task-reward revenue")
    ax.set_title("Cumulative Financial Measures")
    ax.set_xlabel("Period")
    ax.set_ylabel("Amount ($)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[2, 2]
    ax.bar(["Stockout events", "Uncompleted requests"], [results["stockout_events"], results["request_count"] - results["completed_tasks"]], color=["darkred", "darkorange"])
    ax.set_title("Observed Service Disruptions")
    ax.set_ylabel("Count")
    ax.grid(True, axis="y", alpha=0.3)

    fig.tight_layout()
    plt.show()


visualize_mas_results(mas, results, performance_metrics)

In [ ]:
def compare_mas_vs_centralized(mas, results):
    """Summarize observed MAS results without inventing a comparator run."""
    print("\n=== MODEL COMPARISON NOTES ===")
    print("Observed multi-agent simulation:")
    print(f"  Customers: {len(mas.customer_agents)}")
    print(f"  Vehicles: {len(mas.vehicle_agents)}")
    print(f"  Periods: {mas.num_periods}")
    print(f"  Requests completed: {results['completed_tasks']} of {results['request_count']}")
    print(f"  Service level: {results['service_level']:.1f}%")
    print(f"  Total modeled cost: ${results['total_cost']:.2f}")
    print(f"  Net result under the stated accounting convention: ${results['net_profit']:.2f}")
    print(
        "No centralized simulation is run in this notebook, so numerical "
        "claims about relative cost, service, scalability, or optimality "
        "cannot be inferred from these results."
    )
    return {
        "mas_service_level": results["service_level"],
        "mas_cost": results["total_cost"],
        "centralized_benchmark_available": False,
    }


comparison_results = compare_mas_vs_centralized(mas, results)

## Interpreting the Simulation

The printed summaries and plots describe the run produced by the stated input data, random seed, demand model, auction rule, and accounting assumptions. The service level is the cumulative fraction of posted replenishment requests that were completed. A stockout event is counted when sampled demand exceeds a customer's available inventory; it is distinct from an uncompleted delivery request.

The market allocates tasks using local vehicle bids, but this rule does not jointly optimize routes across all vehicles and tasks. Therefore, successful task allocation should not be interpreted as proof of system-wide optimality. Likewise, one simulated run cannot establish general claims about resilience, scalability, or performance relative to a centralized approach. Such claims require suitable experiments and a clearly specified comparator.

### Agent roles in this implementation

- **Customer agents** sample demand, track inventory, assign request priority, and post replenishment tasks when their inventory falls below the trigger level.
- **Vehicle agents** check capacity and deadline feasibility, estimate travel-based bids, and execute tasks awarded by the market. Their routes within a period follow the order in which tasks are processed.
- **The depot agent** checks whether sufficient central stock is available, records internal transfer revenue and unit costs, and sets an illustrative inventory- and demand-sensitive price.
- **The market** posts tasks, filters bids using the confidence threshold, awards the lowest eligible bid, and updates a smoothed bid-price indicator after successful completion.

### Limitations and possible extensions

This teaching model omits replenishment of depot stock, explicit communication delays, vehicle breakdowns, stochastic travel times, multi-stop route optimization, and a calibrated inventory policy. Requests that are not fulfilled expire at the end of their bidding period. These assumptions make the coordination sequence inspectable while limiting the conclusions that can be drawn from the simulation.

Natural extensions within the same agent-based framework include adding a depot replenishment process, recording communication or disruption events, and testing additional demand scenarios while reporting the seed and assumptions. Those extensions would require new model inputs and are not implemented here.